# Task 2: Apply Perturbations to ALS-Specific Genes

Utilize the workflow to target previously-reported disease-specific genes which
contribute to pathology in amyotrophic lateral sclerosis (ALS).
Simulate perturbations in sALS and healthy (PN) cells and process the resulting
data using the GeneFormer_V2 (gf-12L-95M-i4096) model to embed the perturbation
effects into a biologically informed latent space.

## Gene selection

The 12 target genes below were selected from Pineda et al. (2024), which
performed transcriptome-wide analysis of upper motor neurons in the primary
motor cortex (MCX) of sALS vs. healthy donors. We draw from three categories:

**Predicted master regulators**, transcription factors inferred to drive the
bulk of differentially expressed genes in L5 VAT1L+ neurons:
`MYT1L`, `REST`, `SREBF2`

**Top transcriptome-wide divergence (TxD) drivers**, genes most strongly
associated with the overall transcriptional divergence score between sALS and PN:
`LYNX1`, `TBR1`

**Axonal/structural ALS-linked genes correlated with TxD**, genes with known
ALS genetics or function that also rank highly in the TxD analysis:
`KIF5A`, `DCTN1`, `DYNC1H1`, `TUBA4A`

**Highly specific DEGs in MCX L5 neurons**, genes with the strongest
differential expression specifically in the affected cell populations:
`NEFH`, `TTC14`, `HSP90AA1`

## Cell type selection

Perturbations are applied to three excitatory neuron subtypes that show the
largest transcriptomic divergence between sALS and PN in the MCX (Pineda et al.
Fig. 4C): `Ex.L5.VAT1L_THSD4`, `Ex.L5.VAT1L_EYA4`, `Ex.L3_L5.SCN4B_NEFH`.
These L5 corticospinal projection neurons are the primary site of upper motor
neuron degeneration in ALS.

## Sampling strategy

Both sALS and PN cells are included in each perturbed dataset. Perturbations are
applied to all cells regardless of disease status. The sALS vs. PN distinction
is used only at analysis time (Task 3) when comparing centroid shifts. Cells are
balanced across groups and cell types (n per stratum capped at 150 or the
smallest available stratum) to avoid any group dominating the embedding.

In [ ]:
# Imports
%load_ext autoreload
%autoreload 2

import gc
import os

import anndata as ad
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from helical.models.geneformer import Geneformer, GeneformerConfig

from src.data import load_data, load_embeddings, save_embeddings
from src.constants import SALS_GENES, RELEVANT_CELLTYPES
from src.utils import sample_cells, get_sample_size
from src.perturbation import perturb_expression

# Prevent MPS from pre-allocating the full GPU memory pool
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

In [ ]:
# Load data
adata = load_data()

In [ ]:
# Sample a balanced subset: n cells per (group × cell type), where n is capped
# at 150 or the smallest stratum, whichever is lower. Cell indices are saved
# to disk so Tasks 3 and 4 use the exact same cells.
print("Sampling cells...")
n = get_sample_size(adata, groups=["SALS", "PN"], celltypes=RELEVANT_CELLTYPES)
print(f"Sampling n={n} cells per (group, celltype)")
adata_sub = sample_cells(adata, groups=["SALS", "PN"], celltypes=RELEVANT_CELLTYPES, n=n)
adata_sub.obs.index.to_series().to_csv("../data/adata_sub_index.csv", index=False)
als_mask_sub = adata_sub.obs["Group"] == "SALS"
print(f"\nFinal subset: {adata_sub.n_obs:,} cells ({als_mask_sub.sum()} sALS, {(~als_mask_sub).sum()} PN)")

In [ ]:
# Build the full list of (gene, direction) pairs to embed.
# Each pair will produce one perturbed AnnData and one embedding matrix.
keys = [(gene, direction) for gene in SALS_GENES for direction in ("down", "up")]

In [ ]:
# Load GeneFormer (gf-12L-95M-i4096), a 12-layer, 95M-parameter transformer
# pretrained on ~30M single-cell transcriptomes. It maps each cell to a 512-dim
# embedding. MPS is used on Apple Silicon; falls back to CPU otherwise.
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

config = GeneformerConfig(model_name="gf-12L-95M-i4096", device=device, batch_size=4)
model = Geneformer(configurer=config)

In [ ]:
# Load any embeddings already computed (allows resuming if the run was interrupted)
embeddings = load_embeddings()

In [ ]:
# Embed the unperturbed baseline. This is the reference embedding against which
# all perturbation shifts are measured in Task 3.
if "baseline" not in embeddings:
    print("Embedding baseline...")
    baseline_dataset = model.process_data(adata_sub)
    embeddings["baseline"] = model.get_embeddings(baseline_dataset)
    del baseline_dataset
    gc.collect()
    assert embeddings["baseline"].shape[0] == adata_sub.n_obs, (
        f"Baseline embedding row count {embeddings['baseline'].shape[0]} "
        f"does not match adata_sub.n_obs {adata_sub.n_obs}"
    )
    save_embeddings(embeddings)
    print(f"Baseline embeddings shape: {embeddings['baseline'].shape}")
else:
    print(f"Skipping baseline (already done). Shape: {embeddings['baseline'].shape}")

In [ ]:
# For each (gene, direction) pair: apply perturbation, embed all cells, save, free memory.
# Embeddings are saved after each gene so a crash only loses the current gene.
# The sALS vs. PN split is not applied here, all cells are perturbed and embedded
# together; the group distinction is used in Task 3 when computing centroid shifts.
with tqdm(keys, desc="Perturbing + embedding") as pbar:
    for gene, direction in pbar:
        key = f"{gene}_{direction}"
        if key in embeddings:
            pbar.set_description(f"{gene} ({direction}) [skip]")
            continue
        pbar.set_description(f"{gene} ({direction})")
        pert = perturb_expression(adata_sub, genes=[gene], direction=direction)
        dataset = model.process_data(pert)
        embeddings[key] = model.get_embeddings(dataset)
        assert embeddings[key].shape[0] == adata_sub.n_obs, (
            f"{key} embedding row count {embeddings[key].shape[0]} "
            f"does not match adata_sub.n_obs {adata_sub.n_obs}"
        )
        del pert, dataset
        gc.collect()
        save_embeddings(embeddings)
        torch.mps.empty_cache()

print(f"\nDone. Embeddings computed for: {list(embeddings.keys())}")